# Stage 7 — Stability Selection (D1 held out)

Complementary-pairs L1 feature selection on the accepted IBD cohorts. Outer-cohort labels are excluded from feature selection and prediction training. An independent signature is also fitted within D1 for disjoint-cohort comparison.

**Execution:** Google Colab CPU. All output files, including **each completed complementary pair**, are written directly to `MyDrive/CMPS355_Research/runs/stability_selection/d1/`. If the Colab runtime stops, reconnect and **Run all** in this same notebook to resume validated pairs. A local internet outage may also disconnect the Colab runtime, so checkpoints—not uninterrupted runtime—are the safeguard.

This is an implementation of complementary-pairs selection frequencies, not the official `stabs` package. A top-50 ranking drives prediction; frequency-threshold signatures are separate research outputs.

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/CMPS355_Research')
assert PROJECT.is_dir(), f'Missing project: {PROJECT}'
DATA_DIR = PROJECT / 'data/validated'
DATASET = DATA_DIR / 'ibd_dataset_sample_aligned.zip'
assert DATASET.is_file(), f'Missing accepted dataset archive: {DATASET}'
OUTPUT_ROOT = PROJECT / 'runs/stability_selection'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_DIR = Path('/content/cmps355_stage7_src')
SOURCE_DIR.mkdir(parents=True,exist_ok=True)
print('Dataset:',DATASET)
print('Output:',OUTPUT_ROOT)
print('Sources:',SOURCE_DIR)

Mounted at /content/drive
Dataset: /content/drive/MyDrive/CMPS355_Research/data/validated/ibd_dataset_sample_aligned.zip
Output: /content/drive/MyDrive/CMPS355_Research/runs/stability_selection
Sources: /content/cmps355_stage7_src


In [2]:
%%writefile /content/cmps355_stage7_src/full_gene_loco.py
"""Independent-cohort full-gene LASSO / Elastic Net nested evaluation."""
from __future__ import annotations

import csv
import hashlib
import io
import json
import os
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

EXPECTED_ZIP_SHA = '8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'
EXPECTED_CSV_SHA = {
    'count_d1_aligned.csv': '0f595746ea572a349580d9d3c0242559345b88c927c161b1982933102e6c812d',
    'count_d2_aligned.csv': '2ff413b813e3e4d10927da7f8342eccb35a00205c319c0eb714388aea97f245d',
    'count_d3_aligned.csv': '33b2e51e682da4c41f53becf00b1fc88bb5710293ce2d9d87837bca86a7b3d48',
    'count_d4_aligned.csv': 'c26d8db65b335c89967bd4e072c61a6b7adcb57715c5fe36107d9bd8cd9409e0',
    'desc_d1.csv': '91aadec65bb7f2c1945e7941222b9f490a0c9041d495b8d22606f614484c7b33',
    'desc_d2.csv': '4198f574167754eb00e007ded8cfc2528d50ead0d80b96c1a5cc8139131a140b',
    'desc_d3.csv': '9db3c39975a50afcbd851d3cdcdf56f1e59e7f9bbe18b3b80fd7328400a31332',
    'desc_d4.csv': 'bdd5f6d5ae9a6713470f1eadab2e62a961a38d417f27401359d55b5d906a1193',
}
COHORTS = ('d1','d2','d3','d4')
C_GRID = (0.001, 0.01, 0.1, 1.0)
ELASTIC_RATIOS = (0.5, 0.9)
MIN_CPM = 1.0
MIN_FRACTION = 0.1
SEED = 355
MAX_ITER = 1600
TOL = 0.001


def sha_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def require(condition: bool, reason: str) -> None:
    if not condition:
        raise ValueError(reason)


def find_input(directory: Path):
    root = Path(directory)
    require(root.exists(), f'Input root not found: {root}')
    unpacked = [p for p in root.rglob('count_d1_aligned.csv') if p.is_file()]
    for file in unpacked:
        folder = file.parent
        if all((folder / name).exists() for name in EXPECTED_CSV_SHA):
            return ('directory', folder)
    archives = [p for p in root.rglob('ibd_dataset_sample_aligned.zip') if p.is_file()]
    require(len(archives) == 1, 'Attach exactly one accepted dataset input containing the eight aligned CSV files')
    return ('archive', archives[0])


def load_data(directory: Path) -> tuple[pd.DataFrame, np.ndarray, tuple[str,...], dict]:
    mode, source = find_input(directory)
    if mode == 'archive':
        require(sha_bytes(source.read_bytes()) == EXPECTED_ZIP_SHA, 'Dataset ZIP checksum mismatch')
        with ZipFile(source) as zf:
            require(zf.testzip() is None, 'ZIP CRC verification failed')
            raw = {name: zf.read(name) for name in EXPECTED_CSV_SHA}
    else:
        raw = {name: (source / name).read_bytes() for name in EXPECTED_CSV_SHA}
    for name, expected in EXPECTED_CSV_SHA.items():
        require(sha_bytes(raw[name]) == expected, f'Dataset file checksum mismatch: {name}')
    headers = {}
    for cohort in COHORTS:
        names = next(csv.reader(io.StringIO(raw[f'count_{cohort}_aligned.csv'].decode('utf-8-sig'))))
        require(len(names) == len(set(names)) and names[0] == 'ID', f'Invalid gene names: {cohort}')
        headers[cohort] = names
    genes = tuple(sorted(set.intersection(*(set(headers[c][1:]) for c in COHORTS))))
    require(len(genes) == 9983, 'Unexpected shared gene count')
    counts = []
    metadata = []
    expected_rows = dict(zip(COHORTS, (260,179,76,174)))
    for cohort in COHORTS:
        desc = pd.read_csv(io.BytesIO(raw[f'desc_{cohort}.csv']), dtype={'ID':'string','Study':'string','class':'string'})
        df = pd.read_csv(io.BytesIO(raw[f'count_{cohort}_aligned.csv']), usecols=['ID',*genes], dtype={'ID':'string'})
        require(len(desc)==len(df)==expected_rows[cohort], f'{cohort}: incorrect sample count')
        require(desc['ID'].tolist()==df['ID'].tolist(), f'{cohort}: sample alignment mismatch')
        require({'ID','Study','class'} <= set(desc.columns), f'{cohort}: missing clinical fields')
        require(set(desc['class'])=={'CD','Not IBD'}, f'{cohort}: unexpected classes')
        mat = df.loc[:, list(genes)].to_numpy(dtype=np.float64, copy=True)
        require(np.isfinite(mat).all() and (mat>=0).all() and (mat.sum(axis=1)>0).all(), f'{cohort}: invalid counts')
        desc = desc[['ID','Study','class']].copy()
        desc.insert(0,'cohort',cohort)
        metadata.append(desc)
        counts.append(mat)
    meta = pd.concat(metadata, ignore_index=True)
    require(meta['ID'].is_unique and len(meta)==689, 'Duplicate or missing patients')
    require(meta['Study'].nunique()==6, 'BioProject count mismatch')
    return meta, np.vstack(counts), genes, {'source_mode':mode,'source_path':str(source), 'zip_sha256':EXPECTED_ZIP_SHA,'csv_sha256':EXPECTED_CSV_SHA}


def log_cpm(values: np.ndarray) -> np.ndarray:
    depths = values.sum(axis=1)
    require(np.all(depths>0), 'Empty library detected')
    return np.log2(1.0 + 1e6 * values / depths[:, None])


def fit_features(train_counts: np.ndarray) -> tuple[np.ndarray, np.ndarray, StandardScaler, dict]:
    x = log_cpm(train_counts)
    threshold = max(2, int(np.ceil(MIN_FRACTION * len(x))))
    valid = ((x > np.log2(1+MIN_CPM)).sum(axis=0) >= threshold) & (x.var(axis=0) > 1e-12)
    indices = np.flatnonzero(valid)
    require(len(indices) >= 2, 'Insufficient genes after training-only filtering')
    scaler = StandardScaler()
    fitted = scaler.fit_transform(x[:, indices])
    require(np.isfinite(fitted).all(), 'Invalid prepared features')
    return fitted, indices, scaler, {'fit_patients':len(x), 'prevalence_threshold_samples':threshold,'eligible_genes':len(indices)}


def apply_features(counts: np.ndarray, indices: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    return scaler.transform(log_cpm(counts)[:, indices])


def model_fit(x: np.ndarray, y: np.ndarray, C: float, ratio: float):
    sklearn_major_minor = tuple(int(z) for z in sklearn.__version__.split('.')[:2])
    args = {'l1_ratio':float(ratio)} if sklearn_major_minor >= (1,8) else {
        'penalty':'l1' if ratio==1 else 'elasticnet',
        'l1_ratio': None if ratio==1 else float(ratio)}
    model = LogisticRegression(solver='saga',C=float(C),class_weight='balanced',
                               max_iter=MAX_ITER,tol=TOL,random_state=SEED,**args)
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(x,y)
    converged = not any(issubclass(r.category,ConvergenceWarning) for r in recorded)
    return model, converged


def metrics(y: np.ndarray, prob: np.ndarray) -> dict:
    require(np.isfinite(prob).all() and ((0<=prob)&(prob<=1)).all(), 'Invalid probabilities')
    pred = (prob >= .5).astype(int)
    tn, fp, fn, tp = [int(v) for v in confusion_matrix(y,pred,labels=[0,1]).ravel()]
    return dict(auroc=float(roc_auc_score(y,prob)), average_precision=float(average_precision_score(y,prob)),
                balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                sensitivity=tp/(tp+fn), specificity=tn/(tn+fp),
                tp=tp,tn=tn,fp=fp,fn=fn)


def safe_csv(df: pd.DataFrame, path: Path):
    temporary = path.with_suffix(path.suffix+'.tmp')
    df.to_csv(temporary, index=False)
    os.replace(temporary,path)


def safe_json(obj: dict, path: Path):
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(obj,indent=2,sort_keys=True),encoding='utf-8')
    os.replace(temporary,path)


def run(heldout: str, input_root: Path, output_root: Path) -> Path:
    require(heldout in COHORTS, 'Unknown outer cohort')
    started = time.monotonic()
    meta,counts,genes,source=load_data(input_root)
    y = (meta['class']=='CD').to_numpy(dtype=int)
    cohorts=meta['cohort'].to_numpy()
    test_ids=np.flatnonzero(cohorts==heldout)
    train_ids=np.flatnonzero(cohorts!=heldout)
    require(len(test_ids)>0 and len(train_ids)+len(test_ids)==689, 'Outer fold coverage invalid')
    target=Path(output_root) / heldout
    require(not (target/'report.json').exists(), 'Existing completed results cannot be overwritten')
    target.mkdir(parents=True,exist_ok=True)
    scores=[]
    selected=[]
    predictions=[]
    methods={ 'lasso':(1.0,), 'elastic_net':ELASTIC_RATIOS }
    output={ 'status':'RUNNING', 'heldout':heldout,'dataset':source, 'config':{
        'folds':'nested leave-one-analytical-cohort-out', 'gene_universe':'9983 genes shared before inspecting labels',
        'normalization':'log2(1 + CPM) per sample on shared gene universe',
        'filter':'training-only CPM > 1 in >=10% of training samples, nonzero variance',
        'scaling':'training-only standardization','outer_score_threshold':0.5,
        'tuning_metric':'mean of 3 independent cohort-validation AUROCs',
        'C_grid':C_GRID, 'elastic_ratios':ELASTIC_RATIOS,'max_iter':MAX_ITER,'tol':TOL,
        'class_weight':'balanced','random_seed':SEED}, 'software':{
        'python':platform.python_version(), 'numpy':np.__version__,'pandas':pd.__version__,
        'scipy':scipy.__version__,'sklearn':sklearn.__version__},
        'candidates':[], 'results':[], 'start_utc':datetime.now(timezone.utc).isoformat() }
    safe_json(output,target/'progress.json')
    for method, ratios in methods.items():
        configs=[(C,r) for r in ratios for C in C_GRID]
        candidate_scores={ (float(c),float(r)):[] for c,r in configs }
        for inner in COHORTS:
            if inner == heldout:
                continue
            ix=np.flatnonzero((cohorts!=heldout)&(cohorts!=inner))
            vx=np.flatnonzero(cohorts==inner)
            require(np.intersect1d(np.r_[ix,vx],test_ids).size==0, 'Outer test leakage')
            require(np.intersect1d(ix,vx).size==0, 'Inner sample overlap')
            train_matrix, inds, scaler, prep=fit_features(counts[ix])
            val_matrix=apply_features(counts[vx],inds,scaler)
            print(f'{heldout}/{method}: validating on {inner} using {len(inds)} genes', flush=True)
            for c,r in configs:
                model, conv = model_fit(train_matrix,y[ix],c,r)
                prob=model.predict_proba(val_matrix)[:,1]
                score=metrics(y[vx],prob)
                scores.append(dict(outer=heldout,method=method,validation=inner,C=c,l1_ratio=r,
                                   n_fit=len(ix),n_validation=len(vx),
                                   n_candidate_genes=len(inds), nonzero=int((np.abs(model.coef_[0])>1e-8).sum()),
                                   converged=conv, n_iter=int(model.n_iter_[0]),**score))
                candidate_scores[(float(c),float(r))].append(score['auroc'])
        require(all(len(z)==3 for z in candidate_scores.values()), 'Incomplete inner evaluation')
        ranked=sorted(candidate_scores,key=lambda cr:(-np.mean(candidate_scores[cr]),cr[0],-cr[1]))
        c,r=ranked[0]
        all_x, inds,scaler,preprocess=fit_features(counts[train_ids])
        heldout_x=apply_features(counts[test_ids],inds,scaler)
        final,conv=model_fit(all_x,y[train_ids],c,r)
        pr=final.predict_proba(heldout_x)[:,1]
        result=dict(method=method,outer=heldout,C=c,l1_ratio=r,
                    inner_mean_auroc=float(np.mean(candidate_scores[(c,r)])),
                    train_patients=len(train_ids),test_patients=len(test_ids),
                    candidate_gene_count=len(inds),nonzero_coefficients=int((np.abs(final.coef_[0])>1e-8).sum()),
                    converged=conv,n_iter=int(final.n_iter_[0]),preprocessing=preprocess,**metrics(y[test_ids],pr))
        output['results'].append(result)
        for i,prob in zip(test_ids,pr):
            predictions.append(dict(outer=heldout,method=method,ID=str(meta.iloc[i]['ID']),
                                    Study=str(meta.iloc[i]['Study']),class_true=str(meta.iloc[i]['class']),
                                    y_true=int(y[i]),probability_CD=float(prob)))
        for position in np.flatnonzero(np.abs(final.coef_[0])>1e-8):
            selected.append(dict(outer=heldout,method=method,gene=genes[inds[position]],
                                 coef_scaled=float(final.coef_[0,position])))
        output['candidates'].extend([dict(method=method,C=z[0],l1_ratio=z[1],
                               mean_validation_auroc=float(np.mean(candidate_scores[z]))) for z in ranked])
        safe_csv(pd.DataFrame(scores),target/'inner_scores.csv')
        safe_csv(pd.DataFrame(predictions),target/'predictions.csv')
        safe_csv(pd.DataFrame(selected,columns=['outer','method','gene','coef_scaled']),target/'selected_genes.csv')
        safe_json(output,target/'progress.json')
        print(f"{heldout}/{method}: AUROC={result['auroc']:.3f}, selected={result['nonzero_coefficients']}, converged={conv}",flush=True)
    require(len(output['results'])==2 and len(predictions)==2*len(test_ids),'Incomplete outer fold')
    require(all(len({x['ID'] for x in predictions if x['method']==m})==len(test_ids) for m in methods),
            'Duplicate outer predictions')
    output['status']='PASS' if all(item['converged'] for item in output['results']) and all(s['converged'] for s in scores) else 'WARN_CONVERGENCE'
    output['elapsed_seconds']=round(time.monotonic()-started,2)
    output['completed_utc']=datetime.now(timezone.utc).isoformat()
    output['artifacts_sha256']={f:hashlib.sha256((target/f).read_bytes()).hexdigest()
                                for f in ('inner_scores.csv','predictions.csv','selected_genes.csv')}
    safe_json(output,target/'report.json')
    print(f'FINAL_STATUS={output["status"]}; HELDOUT={heldout}; ELAPSED_SECONDS={output["elapsed_seconds"]}',flush=True)
    return target


Writing /content/cmps355_stage7_src/full_gene_loco.py


In [3]:
%%writefile /content/cmps355_stage7_src/stability_selection.py
"""Cohort-aware complementary-pairs stability selection benchmark.

Method: half-sample selection frequencies from an L1 logistic base learner,
following the stability-selection principle of Meinshausen and Buehlmann (2010).
This is a transparent original implementation, not the CRAN stabs package.
This variant persists validated, atomically-written pair checkpoints on Google Drive.
"""
from __future__ import annotations

import hashlib
import json
import os
import shutil
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

from full_gene_loco import (
    EXPECTED_ZIP_SHA, COHORTS, fit_features, apply_features, load_data,
    metrics, require, safe_csv, safe_json,
)

SEED = 7355
N_COMPLEMENTARY_PAIRS = 30
TARGET_BASE_Q = 50
PREDICTION_TOP_K = 50
PRIMARY_CUTOFF = 0.75
C_GRID = tuple(float(c) for c in np.geomspace(0.0003, 3, 15))
CUTOFFS = (0.60, 0.75, 0.90)
MAX_ITER = 1500
TOL = 1e-4


def announce(message: str, log_path: Path) -> None:
    print(message, flush=True)
    with Path(log_path).open('a', encoding='utf-8') as handle:
        handle.write(message + '\n')


def sha(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def atomic_npz(path: Path, **kwargs) -> None:
    tmp = path.with_suffix('.tmp.npz')
    np.savez_compressed(tmp, **kwargs)
    os.replace(tmp, path)


def fit_sparse(x: np.ndarray, y: np.ndarray, C: float, seed: int):
    require(len(np.unique(y)) == 2, 'Half sample has only one class')
    model = LogisticRegression(
        penalty='l1', solver='liblinear', C=float(C),
        class_weight='balanced', max_iter=MAX_ITER, tol=TOL,
        random_state=int(seed),
    )
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(x, y)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in observed)
    coef = np.asarray(model.coef_[0], dtype=np.float64)
    require(np.isfinite(coef).all(), 'Nonfinite coefficients')
    return coef, converged, int(model.n_iter_[0])


def calibrate(x: np.ndarray, y: np.ndarray, seed: int):
    rng = np.random.default_rng(seed)
    half = rng.choice(len(y), size=len(y)//2, replace=False)
    records = []
    for j,C in enumerate(C_GRID):
        coef,conv,iters = fit_sparse(x[half], y[half], C, seed + j + 1)
        records.append(dict(C=float(C), base_selected=int(np.count_nonzero(coef)),
                            converged=bool(conv), n_iter=iters))
    # Calibrate once using training labels; no outcome or information from outer test.
    # Prefer a nonzero solution close to the target sparsity, breaking ties toward fewer genes.
    eligible = [r for r in records if r['base_selected'] > 0 and r['converged']]
    require(len(eligible) > 0, 'No converged nonempty calibration candidate; inspect C grid')
    chosen = min(eligible, key=lambda r:(abs(r['base_selected'] - TARGET_BASE_Q),
                                         r['base_selected'], r['C']))
    return chosen['C'], records, half


def _fingerprint_matrix(matrix: np.ndarray) -> str:
    # The exact numeric training matrix must match before a checkpoint can be reused.
    a = np.ascontiguousarray(matrix, dtype=np.float64)
    return hashlib.sha256(a.tobytes()).hexdigest()


def _fingerprint_strings(values) -> str:
    return hashlib.sha256(json.dumps([str(v) for v in values], ensure_ascii=False,
                                     separators=(',', ':')).encode('utf-8')).hexdigest()


def _scope_settings(scope, matrix, labels, sample_ids, gene_names, seed):
    return {
        'schema': 'colab_resumable_pairs_v1',
        'scope': scope,
        'sample_count': int(len(labels)),
        'gene_count': int(matrix.shape[1]),
        'matrix_sha256': _fingerprint_matrix(matrix),
        'labels_sha256': hashlib.sha256(np.ascontiguousarray(labels, dtype=np.int64).tobytes()).hexdigest(),
        'sample_ids_sha256': _fingerprint_strings(sample_ids),
        'gene_names_sha256': _fingerprint_strings(gene_names),
        'source_sha256': sha(Path(__file__)),
        'base_source_sha256': sha(Path(__file__).with_name('full_gene_loco.py')),
        'seed': int(seed),
        'C_grid': list(C_GRID),
        'n_complementary_pairs': N_COMPLEMENTARY_PAIRS,
        'target_base_q': TARGET_BASE_Q,
        'prediction_top_k': PREDICTION_TOP_K,
        'cutoffs': list(CUTOFFS),
        'max_iter': MAX_ITER,
        'tol': TOL,
        'sampling': 'identical uniform complementary halves, unstratified',
    }


def _read_pair_checkpoint(path, pair, n, p, C, seed, labels):
    # A .npz is atomically promoted only after the two fitted halves are complete.
    with np.load(path, allow_pickle=False) as z:
        require(set(z.files) == {
            'pair', 'C', 'seed', 'n', 'p', 'indices0', 'indices1',
            'selected0', 'selected1', 'coef0', 'coef1', 'coef_full0', 'coef_full1', 'iters0', 'iters1',
            'converged0', 'converged1', 'elapsed_seconds'
        }, f'Checkpoint structure mismatch: {path}')
        require(int(z['pair']) == pair and int(z['seed']) == seed and
                int(z['n']) == n and int(z['p']) == p and
                float(z['C']) == float(C), f'Checkpoint configuration mismatch: {path}')
        rng = np.random.default_rng(seed + 100000 + pair)
        permutation = rng.permutation(n)
        expected = (permutation[:n//2], permutation[n//2:])
        halves=[]
        for side in (0, 1):
            ix = np.asarray(z[f'indices{side}'], dtype=int)
            selected = np.asarray(z[f'selected{side}'], dtype=int)
            coefficients = np.asarray(z[f'coef{side}'], dtype=float)
            fullcoef = np.asarray(z[f'coef_full{side}'], dtype=float)
            require(fullcoef.shape==(p,) and np.isfinite(fullcoef).all() and
                    np.array_equal(selected,np.flatnonzero(np.abs(fullcoef)>1e-8)) and
                    np.array_equal(coefficients,fullcoef[selected]),
                    f'Checkpoint dense coefficients mismatch: {path}')
            require(np.array_equal(ix,expected[side]), f'Checkpoint patient mismatch: {path}')
            require(len(selected)==len(coefficients) and
                    (len(selected)==0 or (np.all(np.diff(selected)>0) and
                        selected[0]>=0 and selected[-1]<p)) and
                    np.isfinite(coefficients).all(), f'Checkpoint coefficient mismatch: {path}')
            require(np.unique(labels[ix]).size==2, f'Invalid saved class mix: {path}')
            halves.append((ix, selected, coefficients, fullcoef,
                           bool(z[f'converged{side}']), int(z[f'iters{side}'])))
        elapsed = float(z['elapsed_seconds'])
        require(np.isfinite(elapsed) and elapsed>=0, f'Bad checkpoint timing: {path}')
        return halves,elapsed


def output_scope(*, scope: str, matrix: np.ndarray, labels: np.ndarray,
                 sample_ids: np.ndarray, gene_names: np.ndarray,
                 dest: Path, seed: int, eventlog: Path):
    dest.mkdir(parents=True, exist_ok=True)
    require(np.isfinite(matrix).all(), f'{scope}: nonfinite prepared data')
    require(len(matrix)==len(labels)==len(sample_ids), f'{scope}: mismatch')
    n,p = matrix.shape
    require(n >= 30 and p >= PREDICTION_TOP_K, f'{scope}: insufficient data')
    labels=np.asarray(labels,dtype=int)
    settings=_scope_settings(scope,matrix,labels,sample_ids,gene_names,seed)
    manifest_path=dest/'scope_manifest.json'
    if manifest_path.exists():
        manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
        require(manifest.get('settings')==settings,
                f'{scope}: previous checkpoint provenance differs; refusing unsafe reuse')
        C=float(manifest['selected_C'])
        require(np.isfinite(C) and C>0, f'{scope}: invalid saved C')
        require((dest/'calibration.csv').is_file() and
                (dest/'calibration_membership.csv').is_file(),
                f'{scope}: calibration evidence absent')
        announce(f'{scope}: validated existing calibration; C={C}',eventlog)
    else:
        require(not (dest/'checkpoints').exists(),
                f'{scope}: found pair checkpoints without scope_manifest.json; investigate')
        c0=time.monotonic()
        C, calibration, calibration_ix = calibrate(matrix, labels, seed)
        safe_csv(pd.DataFrame(calibration), dest/'calibration.csv')
        safe_csv(pd.DataFrame({'scope':scope, 'ID':[str(sample_ids[i]) for i in calibration_ix],
                               'class_binary':labels[calibration_ix]}),dest/'calibration_membership.csv')
        safe_json(dict(settings=settings,selected_C=C,
                       calibration_seconds=round(time.monotonic()-c0,3)),manifest_path)
        announce(f'{scope}: calibration saved; C={C}',eventlog)
    cpdir=dest/'checkpoints'
    cpdir.mkdir(parents=True,exist_ok=True)
    for incomplete in cpdir.glob('*.tmp*'):
        incomplete.unlink()
    existing=list(cpdir.glob('pair_*.npz'))
    require(all(f'pair_{i:03d}.npz' in {f.name for f in existing}
                for i in range(len(existing))) and len(existing)<=N_COMPLEMENTARY_PAIRS,
            f'{scope}: unexpected or non-contiguous pair checkpoints')
    if existing:
        announce(f'{scope}: resuming from {len(existing)}/{N_COMPLEMENTARY_PAIRS} saved pairs',eventlog)
    counts=np.zeros(p,dtype=np.int64)
    abs_sums=np.zeros(p,dtype=np.float64)
    fit_rows=[]
    memberships=[]
    sparse_records=[]
    n_nonconverged=0
    total_fit_seconds=0.0
    t0=time.monotonic()
    for pair in range(N_COMPLEMENTARY_PAIRS):
        cp=cpdir/f'pair_{pair:03d}.npz'
        if not cp.exists():
            cstart=time.monotonic()
            rng = np.random.default_rng(seed+100000+pair)
            perm=rng.permutation(n)
            halves=(perm[:n//2],perm[n//2:])
            require(len(set(halves[0]) & set(halves[1]))==0,
                    'Complementary sample overlap')
            out=[]
            for side,ix in enumerate(halves):
                require(np.unique(labels[ix]).size==2,'Invalid half-sample class mix')
                coef,conv,iters=fit_sparse(matrix[ix],labels[ix],C,
                                           seed+200000+2*pair+side)
                sel=np.flatnonzero(np.abs(coef)>1e-8)
                out.append((np.asarray(ix,dtype=np.int64),sel.astype(np.int32),
                            coef[sel].astype(np.float64),coef.astype(np.float64),
                            bool(conv),int(iters)))
            atomic_npz(cp,pair=np.int64(pair),C=np.float64(C),seed=np.int64(seed),
                       n=np.int64(n),p=np.int64(p),
                       indices0=out[0][0],indices1=out[1][0],
                       selected0=out[0][1],selected1=out[1][1],
                       coef0=out[0][2],coef1=out[1][2],
                       coef_full0=out[0][3],coef_full1=out[1][3],
                       converged0=np.bool_(out[0][4]),converged1=np.bool_(out[1][4]),
                       iters0=np.int64(out[0][5]),iters1=np.int64(out[1][5]),
                       elapsed_seconds=np.float64(time.monotonic()-cstart))
        halves,elapsed=_read_pair_checkpoint(cp,pair,n,p,C,seed,labels)
        total_fit_seconds+=elapsed
        for side,(ix,selected,coefficients,fullcoef,conv,iters) in enumerate(halves):
            counts[selected]+=1
            abs_sums+=np.abs(fullcoef)
            n_nonconverged+=int(not conv)
            sparse_records.extend(dict(scope=scope,pair=pair,half=side,
                                       gene=str(gene_names[j]),coef=float(value))
                                  for j,value in zip(selected,coefficients))
            fit_rows.append(dict(scope=scope,pair=pair,half=side,n_samples=len(ix),
                                 n_cd=int(labels[ix].sum()),n_controls=int(len(ix)-labels[ix].sum()),
                                 selected_genes=len(selected),converged=conv,n_iter=iters,C=C))
            memberships.extend(dict(scope=scope,pair=pair,half=side,ID=str(sample_ids[i]))
                               for i in ix)
        # The pair .npz is the durable resume point, stored directly on Drive.
        safe_json(dict(status='RUNNING',scope=scope,completed_pairs=pair+1,
                       total_pairs=N_COMPLEMENTARY_PAIRS,
                       fit_seconds_completed=round(total_fit_seconds,2)),dest/'progress.json')
        if (pair+1)%5==0 or pair+1==N_COMPLEMENTARY_PAIRS:
            atomic_npz(dest/'checkpoint_counts.npz',completed_pairs=np.int64(pair+1),
                       selection_counts=counts,abs_coef_sums=abs_sums)
            safe_csv(pd.DataFrame(fit_rows),dest/'subsample_fits.csv')
            safe_csv(pd.DataFrame(memberships),dest/'subsample_membership.csv')
            safe_csv(pd.DataFrame(sparse_records,
                     columns=['scope','pair','half','gene','coef']),
                     dest/'subsample_nonzero_coefficients.csv')
            announce(f'{scope}: {pair+1}/{N_COMPLEMENTARY_PAIRS} complementary pairs completed',eventlog)
    total=2*N_COMPLEMENTARY_PAIRS
    freq=counts/total
    mean_abs=abs_sums/total
    order=sorted(range(p),key=lambda j:(-freq[j],-mean_abs[j],str(gene_names[j])))
    ranking=pd.DataFrame({
        'scope':scope,'gene':[str(gene_names[j]) for j in order],
        'rank':np.arange(1,p+1,dtype=int),
        'selection_count':counts[order],
        'selection_frequency':freq[order],
        'mean_abs_subsample_coefficient':mean_abs[order],
        'top50_for_prediction':np.arange(1,p+1)<=PREDICTION_TOP_K,
        **{f'selected_pi_{str(c).replace(".", "_")}':freq[order]>=c for c in CUTOFFS}
    })
    safe_csv(ranking,dest/'gene_ranking.csv')
    subsets=[]
    for c in CUTOFFS:
        sel=ranking.loc[ranking['selection_frequency']>=c,
                        ['gene','rank','selection_frequency']].copy()
        sel['cutoff']=c
        subsets.append(sel)
    safe_csv(pd.concat(subsets,ignore_index=True),dest/'threshold_signatures.csv')
    q_mean=float(np.mean([r['selected_genes'] for r in fit_rows]))
    diagnostics={
        'scope':scope,'seed':seed,'n_samples':n,'eligible_gene_count':p,
        'sampling':'uniform random complementary halves; no outcome stratification',
        'selection_base':'L1-penalized logistic regression; class_weight=balanced; liblinear',
        'calibration':'one training-only half-sample, C nearest target q=50 with converged nonzero fit',
        'target_base_q':TARGET_BASE_Q,'selected_C':C,
        'n_complementary_pairs':N_COMPLEMENTARY_PAIRS,'n_halves':total,
        'average_base_selected_q':q_mean,'nonconverged_half_fits':n_nonconverged,
        'cutoff_signature_sizes':{str(c):int((freq>=c).sum()) for c in CUTOFFS},
        'top50_signature_size':int(min(PREDICTION_TOP_K,np.count_nonzero(freq>0))),
        'mb_2010_bound_expression_at_primary_cutoff':float(q_mean*q_mean/((2*PRIMARY_CUTOFF-1)*p)),
        'bound_caution':('Numerical Meinshausen-Buehlmann bound expression only, not a '
                         'certified finite-sample error guarantee; exchangeable nulls, '
                         'base selector assumptions, and calibration conditions are unverified.'),
        'elapsed_seconds':round(time.monotonic()-t0,2),
        'fit_seconds_across_all_attempts':round(total_fit_seconds,2),
        'durable_pair_checkpoints':str(cpdir),
        'checkpoint_pairs_validated':N_COMPLEMENTARY_PAIRS,
    }
    safe_json(diagnostics,dest/'selection_report.json')
    safe_json(dict(status='COMPLETE',scope=scope,completed_pairs=N_COMPLEMENTARY_PAIRS,
                   fit_seconds_completed=round(total_fit_seconds,2)),dest/'progress.json')
    return order,freq,diagnostics

def run(heldout: str, input_root: Path, output_root: Path) -> Path:
    require(heldout in COHORTS, 'Unknown cohort')
    t0=time.monotonic()
    meta, raw_counts, genes, data_source=load_data(Path(input_root))
    y=(meta['class']=='CD').to_numpy(dtype=int)
    cohorts=meta['cohort'].to_numpy()
    tr=np.flatnonzero(cohorts!=heldout)
    te=np.flatnonzero(cohorts==heldout)
    require(len(tr)+len(te)==689 and np.intersect1d(tr,te).size==0, 'Invalid LOCO')
    dest=Path(output_root)/heldout
    dest.mkdir(parents=True,exist_ok=True)
    finished=dest/'report.json'
    if finished.is_file():
        prior=json.loads(finished.read_text(encoding='utf-8'))
        require(prior.get('heldout')==heldout and
                prior.get('expected_dataset_sha256')==EXPECTED_ZIP_SHA and
                prior.get('status') in ('PASS','WARN_CONVERGENCE'),
                'Existing report is not a valid completed result')
        for relative, expected_sha in prior.get('artifacts_sha256',{}).items():
            artifact=dest/relative
            require(artifact.is_file() and sha(artifact)==expected_sha,
                    f'Completed run integrity failed: {relative}')
        print(f'ALREADY_COMPLETE heldout={heldout} status={prior["status"]}',flush=True)
        return dest
    require(not (dest/'run_manifest.json').exists() and not (dest/'progress.json').exists(),
            'Previous progress exists but run_manifest.json is missing; investigate before resuming')
    manifest_path=dest/'run_manifest.json'
    settings={'schema':'colab_resume_v1','heldout':heldout,
              'dataset_sha256':EXPECTED_ZIP_SHA,'source_sha256':sha(Path(__file__)),
              'base_source_sha256':sha(Path(__file__).with_name('full_gene_loco.py')),
              'train_patients':len(tr),'test_patients':len(te)}
    if manifest_path.exists():
        manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
        require(manifest.get('settings')==settings,
                'Existing run configuration/source differs; cannot safely resume')
        start=manifest['start_utc']
        announce(f'RESUME {datetime.now(timezone.utc).isoformat()} heldout={heldout}',dest/'events.log')
    else:
        start=datetime.now(timezone.utc).isoformat()
        safe_json(dict(settings=settings,start_utc=start),manifest_path)
        announce(f'START {start} heldout={heldout}',dest/'events.log')
    safe_json(dict(status='RUNNING',heldout=heldout,start_utc=start),dest/'progress.json')
    try:
        # PART A: all feature selection and predictor training use only the other 3 cohorts.
        tr_x, tr_genes, scaler, prep=fit_features(raw_counts[tr])
        te_x=apply_features(raw_counts[te],tr_genes,scaler)
        outer_names=np.array(genes,dtype=object)[tr_genes]
        out_order,out_freq,out_diag=output_scope(
            scope='loco_training',matrix=tr_x,labels=y[tr],
            sample_ids=meta.iloc[tr]['ID'].to_numpy(),gene_names=outer_names,
            dest=dest/'loco_training',seed=SEED+COHORTS.index(heldout)*100,eventlog=dest/'events.log')
        # Prediction uses a PREDECLARED top-50 ranking budget; cutoff signatures are reported separately.
        top=np.array([j for j in out_order if out_freq[j]>0][:PREDICTION_TOP_K],dtype=int)
        require(len(top)>=2, 'Insufficient selected genes for predictive baseline')
        clf=LogisticRegression(penalty='l2',solver='liblinear',class_weight='balanced',
                               C=1.0,max_iter=2000,random_state=SEED)
        clf.fit(tr_x[:,top], y[tr])
        prob=clf.predict_proba(te_x[:,top])[:,1]
        perf=metrics(y[te],prob)
        safe_csv(pd.DataFrame([{
            'outer':heldout,'method':'complementary_pairs_l1_stability_top50',
            'ID':str(meta.iloc[i]['ID']),'Study':str(meta.iloc[i]['Study']),
            'class_true':str(meta.iloc[i]['class']),'y_true':int(y[i]),
            'probability_CD':float(pr)} for i,pr in zip(te,prob)]),dest/'predictions.csv')
        safe_csv(pd.DataFrame(dict(rank=np.arange(1,len(top)+1),gene=outer_names[top],
                                  selection_frequency=out_freq[top],
                                  prediction_classifier_coef=clf.coef_[0])),dest/'prediction_signature.csv')
        atomic_npz(dest/'prediction_model.npz', selected_global_gene_indices=tr_genes[top],
                   selected_genes=outer_names[top].astype(str),
                   scaler_mean=scaler.mean_[top],scaler_scale=scaler.scale_[top],
                   coef=clf.coef_[0],intercept=clf.intercept_,
                   decision_threshold=np.array([0.5]))
        announce(f'{heldout}: held-out AUROC={perf["auroc"]:.3f}; predictions saved',dest/'events.log')
        # PART B: disjoint-cohort signature analysis, isolated from the predictive procedure.
        # Held-out cohort labels are used ONLY after Part A predictions are already saved.
        single_x,sg,scaler2,sprep=fit_features(raw_counts[te])
        single_names=np.array(genes,dtype=object)[sg]
        ind_order,ind_freq,ind_diag=output_scope(
            scope='independent_cohort',matrix=single_x,labels=y[te],
            sample_ids=meta.iloc[te]['ID'].to_numpy(),gene_names=single_names,
            dest=dest/'independent_cohort',seed=SEED+1000+COHORTS.index(heldout)*100,eventlog=dest/'events.log')
        report={
            'status':'PASS' if out_diag['nonconverged_half_fits']==0 and ind_diag['nonconverged_half_fits']==0 else 'WARN_CONVERGENCE',
            'method':'complementary_pairs_l1_stability_selection',
            'method_fidelity':('Transparent MB-2010-inspired half-sample frequencies, '
                               'not a run of the official R stabs implementation'),
            'heldout':heldout,'start_utc':start,'end_utc':datetime.now(timezone.utc).isoformat(),
            'dataset':data_source,'expected_dataset_sha256':EXPECTED_ZIP_SHA,
            'n_train':len(tr),'n_test':len(te),
            'train_cohorts':[c for c in COHORTS if c!=heldout],
            'test_cohort':heldout,
            'protocol':{
                'gene_universe':'9983 genes shared across four cohorts',
                'normalization':'log2(1+CPM) based on counts in shared genes',
                'gene_filter':'train-only CPM>1 in >=10% and nonzero variance',
                'scaler':'train-only standardization',
                'selection_base':'L1 logistic, calibrated C for approximately q=50 base selections',
                'sampling':'30 complementary random half-sample pairs (60 fits) per scope',
                'primary_cutoff':PRIMARY_CUTOFF,
                'other_cutoffs':[c for c in CUTOFFS if c!=PRIMARY_CUTOFF],
                'prediction':'fixed top-50 stability ranking + balanced L2 logistic C=1',
                'prediction_label_leakage_policy':('LOCO predictor fitted only on other three cohorts; '
                    'independent signature on test cohort fitted only after prediction results saved'),
                'no_inner_tuning':'Predictive classifier hyperparameters fixed a priori; differs from nested LASSO/EN protocol',
            },
            'loco_preprocessing':prep,'independent_preprocessing':sprep,
            'outer_selection':out_diag,'independent_selection':ind_diag,
            'prediction_signature_genes':len(top),'prediction_model_n_iter':int(clf.n_iter_[0]),
            'heldout_metrics':perf,
            'checkpoint_policy':'Each completed complementary pair written atomically to Google Drive and verified on resume',
            'software':{'python':platform.python_version(),'numpy':np.__version__,
                        'pandas':pd.__version__,'scipy':scipy.__version__,
                        'sklearn':sklearn.__version__},
            'total_elapsed_seconds':round(time.monotonic()-t0,2),
            'scientific_limits':[
                'A fixed top-50 selection budget is not the primary 0.75 threshold signature.',
                'Selection of a calibrating C is training-only and not part of the stabs author implementation.',
                'No validated false discovery bound should be inferred from reported expression alone.',
                'One LOCO result per held-out cohort, not repeated external validation.',
                'Within-cohort unsupervised scaling precedes resampling; downstream LOCO test data remains unseen.',
                'Gene/pathway overlap must be computed from independent_cohort, not loco_training.'
            ]
        }
        announce(f'FINISHED status={report["status"]} elapsed={report["total_elapsed_seconds"]}',dest/'events.log')
        saved=['predictions.csv','prediction_signature.csv','prediction_model.npz','events.log','run_manifest.json']
        for scope in ('loco_training','independent_cohort'):
            for f in ('calibration.csv','calibration_membership.csv','scope_manifest.json',
                      'gene_ranking.csv','threshold_signatures.csv',
                      'subsample_fits.csv','subsample_membership.csv','subsample_nonzero_coefficients.csv',
                      'checkpoint_counts.npz','selection_report.json','progress.json'):
                saved.append(scope+'/'+f)
        for f in saved:
            require((dest/f).exists(),f'Missing artifact: {f}')
        for script_name in ('full_gene_loco.py','stability_selection.py'):
            source_file=Path(__file__).parent/script_name
            require(source_file.exists(),f'Missing source code: {source_file}')
            shutil.copyfile(source_file,dest/('source_'+script_name))
            saved.append('source_'+script_name)
        for scope in ('loco_training','independent_cohort'):
            for cp in sorted((dest/scope/'checkpoints').glob('pair_*.npz')):
                saved.append(str(cp.relative_to(dest)))
        report['artifacts_sha256']={f:sha(dest/f) for f in saved}
        safe_json(report,dest/'report.json')
        safe_json(dict(status=report['status'],completed_utc=report['end_utc']),dest/'progress.json')
        (dest/'failure.json').unlink(missing_ok=True)
        print(f'FINAL_STATUS={report["status"]}; OUTER={heldout}; ELAPSED_SECONDS={report["total_elapsed_seconds"]}',flush=True)
        return dest
    except Exception as exc:
        safe_json(dict(status='FAIL',heldout=heldout,type=type(exc).__name__,
                       message=str(exc),timestamp_utc=datetime.now(timezone.utc).isoformat()),
                  dest/'failure.json')
        raise


Writing /content/cmps355_stage7_src/stability_selection.py


In [4]:
import json
import sys
import hashlib
sys.path.insert(0,str(SOURCE_DIR))
from stability_selection import run

output=run(heldout='d1',input_root=DATA_DIR,output_root=OUTPUT_ROOT)
report=json.loads((output/'report.json').read_text(encoding='utf-8'))
print('FINAL_STATUS:', report['status'])
print('HELD_OUT:', report['heldout'])
print('HELD_OUT_AUROC:', report['heldout_metrics']['auroc'])
print('LOCO_0.75_SIGNATURE:', report['outer_selection']['cutoff_signature_sizes']['0.75'])
print('INDEPENDENT_0.75_SIGNATURE:', report['independent_selection']['cutoff_signature_sizes']['0.75'])
print('DIRECT_DRIVE_OUTPUT:', output)

START 2026-11-07T11:27:40.248654+00:00 heldout=d1
loco_training: calibration saved; C=0.11182781160944819
loco_training: 5/30 complementary pairs completed
loco_training: 10/30 complementary pairs completed
loco_training: 15/30 complementary pairs completed
loco_training: 20/30 complementary pairs completed
loco_training: 25/30 complementary pairs completed
loco_training: 30/30 complementary pairs completed
d1: held-out AUROC=0.826; predictions saved
independent_cohort: calibration saved; C=0.8048087385839177
independent_cohort: 5/30 complementary pairs completed
independent_cohort: 10/30 complementary pairs completed
independent_cohort: 15/30 complementary pairs completed
independent_cohort: 20/30 complementary pairs completed
independent_cohort: 25/30 complementary pairs completed
independent_cohort: 30/30 complementary pairs completed
FINISHED status=PASS elapsed=35.47
FINAL_STATUS=PASS; OUTER=d1; ELAPSED_SECONDS=35.47
FINAL_STATUS: PASS
HELD_OUT: d1
HELD_OUT_AUROC: 0.82557885539536